In [73]:
import csv
import pandas as pd
import re

In [2]:
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

In [3]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"

last_file = "anonymous_answer_2026-07-24.csv"


In [4]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/{last_file}", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
df_col = df_col.iloc[:143]
df0

,qno_obs,q1_so_principles,q2_hal_depot,q3_octavi_depot,q4_diff_data,q4_autres_entrepots,q5_r,q5_python,q5_excel,q5_julia,...,q45_clé,q46_date_de_saisie,q47_date_de_dernière_modification,q48_date_de_dernier_enregistrement,q49_temps_de_saisie,q50_langue,q51_progression,q52_dernière_question_saisie,q53_origine,q54_appareil_utilisé_pour_la_saisie
0,1,"Oui, un peu","Oui, une fois",Non,Non,NaN,Non,Non,Non,Non,...,PS8W-TJ9P,09/03/2026 11:01:53,NaN,09/03/2026 11:12:04,"611,0190107",FR,Terminé,NaN,Direct,PC
1,2,"Oui, un peu","Oui, une fois",Non,Non,NaN,Non,Non,Non,Non,...,GNEG-5QXQ,10/03/2026 11:11:19,NaN,10/03/2026 11:17:47,"387,3420906",FR,Terminé,NaN,Direct,PC
2,3,"Oui, tout à fait","Oui, plusieurs fois",Je ne connais pas Octaviana,Non,NaN,Non,Non,Oui,Non,...,FCCB-GMYA,10/03/2026 11:10:00,NaN,10/03/2026 11:19:57,"596,9405636",FR,Terminé,NaN,Direct,PC
3,4,"Oui, un peu","Oui, une fois",Non,"Autre, précisez",OSF,Oui,Non,Non,Non,...,CV3G-95CT,10/03/2026 11:14:07,NaN,10/03/2026 11:23:25,"558,0371672",FR,Terminé,NaN,Direct,PC
4,5,"Oui, un peu","Oui, systématiquement",Non,Non,NaN,Non,Non,Oui,Non,...,9EAP-NB4B,10/03/2026 11:18:54,NaN,10/03/2026 11:28:09,"554,5038942",FR,Terminé,NaN,Direct,PC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,117,"Oui, un peu","Oui, plusieurs fois",Non,"Autre, précisez",OSF,Oui,Oui,Oui,Non,...,MLSN-NBUS,28/04/2026 17:33:55,NaN,28/04/2026 17:49:27,"932,1846343",FR,Terminé,NaN,Direct,PC
117,118,"Oui, un peu","Oui, systématiquement",Non,Non,NaN,Oui,Non,Oui,Non,...,WAR9-LFJP,29/04/2026 14:59:02,NaN,29/04/2026 15:10:56,"714,0441946",FR,Terminé,NaN,Direct,PC
118,119,"Oui, un peu","Oui, plusieurs fois",Non,Non,NaN,Oui,Non,Oui,Non,...,HRZG-6JGE,05/05/2026 09:36:22,NaN,05/05/2026 09:52:15,"953,6974524",FR,Terminé,NaN,Direct,PC
119,120,Non,"Oui, plusieurs fois",Non;Je ne connais pas Octaviana,"Autre, précisez",Mendeley data,Non,Oui,Oui,Non,...,32TW-JLY5,28/05/2026 10:01:22,NaN,28/05/2026 10:21:58,"1235,8811168",FR,Terminé,NaN,Direct,PC


In [5]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df_split.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).sort_values("nb", ascending=False).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100
    
    return df_explode, gb_data
    

In [6]:
def grouped_question(data, column, index = "q45_clé"):
    """


    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_tmp = data.copy()
    gb_data = df_tmp.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data

In [7]:
def tab_croise(data, x, y, regroup_y = True, khideux = False) :
    """
    x = variable en ligne
    y = variable en colonne
    """
    if regroup_y == True:

        data.loc[data[y].str.lower().str.contains("oui"), f"{y}_rec"] = "Oui"
        data.loc[data[y].str.lower().str.contains("non"), f"{y}_rec"] = "Non"

        cross_tab = pd.crosstab(data[x], data[f"{y}_rec"], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[f"{y}_rec"], margins = False)
    else:
        cross_tab = pd.crosstab(data[x], data[y], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[y], margins = False, normalize=False)

    if khideux == True:
        print(scipy.stats.chi2_contingency(cross_tab0))
        st_chi2, st_p, st_dof, st_exp = scipy.stats.chi2_contingency(cross_tab0)
        chi2 = pd.DataFrame(data={"stats":["chi2","df","p-value"], "values":[st_chi2, st_dof, st_p]})
        cross_tab = pd.concat([cross_tab, chi2])
    else:
        pass

    return cross_tab.fillna("").reset_index(), cross_tab0

# Recodage variable q25_statut

On recode les personnes qui ont répondu Autre à la question : "Quel est votre statut ?"
Je propose également de faire une autre variable : "Doctorant/non doctorant"

In [8]:
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"chercheur")), "q25_statut_rec"] = "Chercheur"
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"doctorante|docteur")), "q25_statut_rec"] = "Doctorant"
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"ing")), "q25_statut_rec"] = "Personnel d'appui à la recherche"

df0.loc[df0.q25_statut=="Doctorant·e", "q25_statut_rec"] = "Doctorant"
df0.loc[df0.q25_statut=="Enseignant·e chercheur·e", "q25_statut_rec"] = "Chercheur"
df0.loc[df0.q25_statut=="Personnel de soutien à la recherche", "q25_statut_rec"] = "Personnel d'appui à la recherche"
df0.q25_statut_rec.value_counts()

q25_statut_rec
Chercheur                           106
Doctorant                            12
Personnel d'appui à la recherche      3
Name: count, dtype: int64

In [9]:
dic_new_var = {'name':'110. statut_rec', 
               'label':'q25_statut_rec', 
               'group':'12_descript_repondant', 
               'personal_data':False, 
               'type':'simple_nominal', 
               'opened_question':False,
               'type_panda':df0.q25_statut_rec.dtypes,
               'no_question':25,
               'question':df_col.question.loc[df_col.label=="q25_statut"].iloc[0]
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])

new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)


In [10]:
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"chercheur")), "q25_doctorant"] = False
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"doctorante|docteur")), "q25_doctorant"] = True
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"ing")), "q25_doctorant"] = False

df0.loc[df0.q25_statut=="Doctorant·e", "q25_doctorant"] = True
df0.loc[df0.q25_statut=="Enseignant·e chercheur·e", "q25_doctorant"] = False
df0.loc[df0.q25_statut=="Personnel de soutien à la recherche", "q25_doctorant"] = False
df0.q25_doctorant.value_counts()

dic_new_var = {'name':'110. doctorant', 
               'label':'q25_doctorant', 
               'group':'12_descript_repondant', 
               'personal_data':False, 
               'type':'simple_nominal', 
               'opened_question':False,
               'type_panda':df0.q25_doctorant.dtypes,
               'no_question':25,
               'question':df_col.question.loc[df_col.label=="q25_statut"].iloc[0]
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])

new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)


In [11]:
df0.loc[df0.q3_octavi_depot.str.contains("Non|Je ne connais pas"), "q3_octavi_rec"] = "Non"
df0.loc[~df0.q3_octavi_depot.str.contains("Non|Je ne connais pas"), "q3_octavi_rec"] = "Oui"


In [12]:
dic_new_var = {'name':'3. octavi_depot_rec',
               'label':'q3_octavi_rec',
               'group':'2_bib_num',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df0.q3_octavi_rec.dtypes,
               'no_question':3,
               'question':df_col.question.loc[df_col.label=="q3_octavi_depot"].iloc[0]
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)
df_col

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
141,154. Origine,q53_origine,17_metadonnee,False,simple_nominal,False,object,NaN,NaN,q53,NaN
142,155. Appareil utilisé pour la saisie,q54_appareil_utilisé_pour_la_saisie,17_metadonnee,False,simple_nominal,False,object,NaN,NaN,q54,NaN
143,110. statut_rec,q25_statut_rec,12_descript_repondant,False,simple_nominal,False,object,25.0,2/ Êtes-vous ?,NaN,NaN
144,110. doctorant,q25_doctorant,12_descript_repondant,False,simple_nominal,False,object,25.0,2/ Êtes-vous ?,NaN,NaN


In [13]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/{last_file}", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)

In [14]:
df_col.to_csv("../le_questionnaire/dico_variable.csv", sep=",", index= False)

# Production et analyse des données


## Recodage "autres langages"

In [15]:
for x in df0.columns:
    print(x)

qno_obs
q1_so_principles
q2_hal_depot
q3_octavi_depot
q4_diff_data
q4_autres_entrepots
q5_r
q5_python
q5_excel
q5_julia
q5_stata
q5_sas
q5_aucun
q5_autre
q5_1_other_lang
q6_dev_code
q6_1_dev_exemple
q7_software_depot
q7_1_other_sw_depot
q8_depot_hal_help
q8_rediger_pdg_help
q8_respect_rgpd_help
q8_licences_help
q8_depot_donnees_help
q8_metadonnees_help
q8_data_paper_help
q8_fair_data_help
q9_utilite_accomp
q10_type_data
q10_other_type_data
q11_quali_or_quanti
q12_humanum
q12_1_how_humanum
q13_nakala
q13_isidore
q13_sharedocs
q13_hébergement_web
q14_stylo
q14_mattermost
q14_gitlab
q14_matomo
q14_hnurl
q14_kanboard
q15_nakala_help
q15_isidore_help
q15_sharedocs_help
q15_hébergement_web_help
q15_stylo_help
q15_mattermost_help
q15_gitlab_help
q15_matomo_help
q15_hnurl_help
q15_kanboard_help
53. Tableur
54. Logiciels d'analyse statistique (ex. R – Rstudio)
55. Python (web-scraping, traitement automatique des langues…)
56. Logiciels d'exploration de textes, analyse, statistique... (ex. Garga

In [16]:
rec_other_lang ={}
for x in df0.q5_1_other_lang.loc[~df0.q5_1_other_lang.isna()]:
    rec_other_lang[x] = x.lower().replace(", ","|")

rec_other_lang

{'MaxQda, EndNotes, Zotero, ': 'maxqda|endnotes|zotero|',
 'Access': 'access',
 '\u200b-': '\u200b-',
 'Matlab': 'matlab',
 'Zotero': 'zotero',
 'LibreOffice Calc': 'libreoffice calc',
 'csv, google sheets': 'csv|google sheets',
 'Jamovi, Iramuteq, Qualcoder': 'jamovi|iramuteq|qualcoder',
 'Nvivo': 'nvivo',
 'php, javascript, d3.js': 'php|javascript|d3.js',
 'Jamovi': 'jamovi',
 'JAMOVI ': 'jamovi ',
 'c#': 'c#',
 'SPSS, Statistica, Jamovi, JASP': 'spss|statistica|jamovi|jasp',
 'Nvivo, IA': 'nvivo|ia',
 'Nvivo et Atlas.ti': 'nvivo et atlas.ti',
 'SPSS': 'spss',
 'OCaml, C, Bash, PHP': 'ocaml|c|bash|php',
 'CSS, html, xml': 'css|html|xml',
 'LaTeX': 'latex',
 'NVivo': 'nvivo',
 'MatLab': 'matlab',
 "QGIS. J'aimerais utiliser Python, mais il faudrait que je sois formée pour en avoir les bases. J'ai déjà suivi la formation de base de R proposée à P8, qui était bien : reste maintenant à se l'approprier !": "qgis. j'aimerais utiliser python|mais il faudrait que je sois formée pour en avoir

In [17]:
rec_other_lang = {'MaxQda, EndNotes, Zotero, ': 'maxqda|endnotes|zotero',
 '\u200b-': 'aucun',
 'Matlab': 'matlab',
 'Zotero': 'zotero',
 'LibreOffice Calc': 'libreoffice calc',
 'csv, google sheets': 'csv|google sheets',
 'Jamovi, Iramuteq, Qualcoder': 'jamovi|iramuteq|qualcoder',
 'Nvivo': 'nvivo',
 'php, javascript, d3.js': 'php|javascript|d3.js',
 'Jamovi': 'jamovi',
 'JAMOVI ': 'jamovi',
 'c#': 'c#',
 'SPSS, Statistica, Jamovi, JASP': 'spss|statistica|jamovi|jasp',
 'Nvivo, IA': 'nvivo|ia',
 'Nvivo et Atlas.ti': 'nvivo|atlas.ti',
 'SPSS': 'spss',
 'OCaml, C, Bash, PHP': 'ocaml|c/c++|bash|php',
 'CSS, html, xml': 'css|html|xml',
 'LaTeX': 'latex',
 'NVivo': 'nvivo',
 'MatLab': 'matlab',
 "QGIS. J'aimerais utiliser Python, mais il faudrait que je sois formée pour en avoir les bases. J'ai déjà suivi la formation de base de R proposée à P8, qui était bien : reste maintenant à se l'approprier !": "qgis",
 'Access': 'access',
 'File maker pro': 'file maker pro',
 'LibreOffice Calc, Limesurvey, Grist, ActiveTigger, Neo4j, Gephi': 'libreoffice calc|limesurvey|grist|activetigger|neo4j|gephi',
 'Jamovi, SPSS': 'jamovi|spss',
 'C/C++ Cuda Caml Racket Ludii GDL ASP': 'c/c++|cuda|caml|racket|ludii|gdl|asp',
 'R mais avec Iramutex': 'iramuteq',
 'jamovi': 'jamovi',
 'QSR NVivo': 'nvivo',
 'Mplus, SPSS..': 'mplus|spss'}



df0["q5_1_other_lang_rec"]= df0.q5_1_other_lang.map(rec_other_lang.get)

# on enregistre les nouvelles variables dans le dataframe original
#with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "w") as file_out:
#    df0.to_csv(file_out, sep=",", index= False)


In [18]:
dic_new_var = {'name':'14. other_lang_rec',
               'label':'q5_1_other_lang_rec',
               'group':'4_logiciel_traitement',
               'personal_data':False,
               'type':'multiple_choice',
               'opened_question':False,
               'type_panda':df0.q5_1_other_lang_rec.dtypes,
               'no_question':14,
               'question':df_col.question.loc[df_col.label=="q5_1_other_lang"].iloc[0],
               "question_family":df_col.question_family.loc[df_col.label=="q5_1_other_lang"].iloc[0],
               'comment':"Recodage de la question 14. mise en basse casse des logiciels cités."
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)
df_col

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
142,155. Appareil utilisé pour la saisie,q54_appareil_utilisé_pour_la_saisie,17_metadonnee,False,simple_nominal,False,object,NaN,NaN,q54,NaN
143,110. statut_rec,q25_statut_rec,12_descript_repondant,False,simple_nominal,False,object,25.0,2/ Êtes-vous ?,NaN,NaN
144,110. doctorant,q25_doctorant,12_descript_repondant,False,simple_nominal,False,object,25.0,2/ Êtes-vous ?,NaN,NaN
145,3. octavi_depot_rec,q3_octavi_rec,2_bib_num,False,simple_nominal,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,NaN,NaN


In [19]:
list_other_logi = [x for x in df_exp.q5_1_other_lang_rec.unique()]
print(list_other_logi)

NameError: name 'df_exp' is not defined

In [20]:
family_logi = {'maxqda':'CAQDAS', 
               'endnotes':'gestion biblio', 
               'zotero':'gestion biblio', 
               'aucun':'aucun', 
               'matlab':'script et programmation', 
               'libreoffice calc':'feuille de calcul', 
               'csv':'feuille de calcul', 
               'google sheets':'feuille de calcul',
               'jamovi':'logiciels de stats',
               'iramuteq':'analyse de texte', 
               'qualcoder':'CAQDAS',
               'nvivo':'CAQDAS',
               'php':'développement web', 
               'javascript':'développement web',
               'd3.js':'développement web', 
               'c#':'script et programmation', 
               'spss':'logiciels de stats',
               'statistica':'logiciels de stats',
               'jasp':'logiciels de stats',
               'ia':'ia', 
               'atlas.ti':'CAQDAS',
               'ocaml':'script et programmation', 
               'c/c++':'script et programmation',
               'bash':'script et programmation',
               'css':'développement web',
               'html':'développement web', 
               'xml':'développement web',
               'latex':'script et programmation',
               'qgis':'cartographie',
               'access':'gestion de base de donnée',
               'file maker pro':'gestion de base de donnée',
               'limesurvey':'logiciel d\'enquête statistique',
               'grist':'gestion de base de donnée',
               'activetigger':'analyse de texte',
               'neo4j':'gestion de base de donnée',
               'gephi':'analyse de réseau',
               'cuda':'script et programmation', 
               'caml':'script et programmation', 
               'racket':'script et programmation', 
               'ludii':'création de jeu',
               'gdl':'logiciels de stats',
               'asp':'développement web',
               'mplus': 'logiciels de stats',
               'excel': 'feuille de calcul',
               'python': 'script et programmation',
               'r':'script et programmation',
               'stata':'logiciels de stats',
               'julia': 'script et programmation',
               'sas': 'logiciels de stats'
              }

In [21]:
df01 = df0.copy()

dict_logiciel_traitement = {}
for row in df0.q45_clé:
    logiciel_traitement = []
    for x in df_col.label.loc[(df_col.question_family=="q5")& (~df_col.label.str.contains("q5_1"))]:
        if df01[x].loc[df01.q45_clé==row].values == "Oui":
            logiciel_traitement.append(x.lower().replace("q5_",""))
    if len(logiciel_traitement)>0:
        dict_logiciel_traitement[row]= '|'.join(logiciel_traitement)
df01["q5_logiciel_traitement_rec"] = df01.q45_clé.map(dict_logiciel_traitement)



In [22]:

df_exp, gb_data = split_multiple_choices(df01.loc[~df01.q5_logiciel_traitement_rec.isna()], column='q5_logiciel_traitement_rec', index="q45_clé", sep = '|')
df_exp1, gb_data1 = split_multiple_choices(df01.loc[~df01.q5_1_other_lang_rec.isna()], column='q5_1_other_lang_rec', index="q45_clé", sep = '|')

df_exp2 = pd.concat([df_exp[["q45_clé", 'q5_logiciel_traitement_rec']], df_exp1[["q45_clé","q5_1_other_lang_rec"]].rename(columns={"q5_1_other_lang_rec":'q5_logiciel_traitement_rec'})])
df_exp2["q5_family_logi"]= df_exp2.q5_logiciel_traitement_rec.map(family_logi.get)
df_exp3 = df_exp2.loc[~df_exp2.q5_logiciel_traitement_rec.isin(["aucun","autre"])]
df_exp3

,q45_clé,q5_logiciel_traitement_rec,q5_family_logi
2,FCCB-GMYA,excel,feuille de calcul
3,CV3G-95CT,r,script et programmation
4,9EAP-NB4B,excel,feuille de calcul
5,M5L8-UM9F,r,script et programmation
5,M5L8-UM9F,excel,feuille de calcul
...,...,...,...
97,7CU7-JG8K,iramuteq,analyse de texte
106,A7R6-3SWC,jamovi,logiciels de stats
113,Q934-QKAW,nvivo,CAQDAS
114,HKN6-95PQ,mplus,logiciels de stats


In [23]:
df01 = df0.copy()

dict_logiciel_traitement2 = {}
dict_logiciel_fam = {}
for row in df_exp3.q45_clé:
    
    dftmp = df_exp3.loc[df_exp3.q45_clé==row]
    logiciel_traitement2 = [x for x in dftmp.q5_logiciel_traitement_rec.unique()]
    family_logi = [x for x in dftmp.q5_family_logi.unique()]
    dict_logiciel_traitement2[row] = "|".join(logiciel_traitement2)
    dict_logiciel_fam[row] = "|".join(family_logi)


In [24]:
df0["q5_logiciel_traitement_rec"]= df0.q45_clé.map(dict_logiciel_traitement2.get)
df0["q5_family_logi"] = df0.q45_clé.map(dict_logiciel_fam.get)


# on enregistre les nouvelles variables dans le dataframe original
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/{last_file}.csv", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)

#On enregistre à côté la liste des familles de logiciel. Permettra de "mapper" plus facilement le nom des logiciels avec leurs familles pour les traitements ultérieurs
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/type_of_software.csv", "w") as file_out:
    df_exp2.drop(columns=["q45_clé"]).drop_duplicates().to_csv(file_out, sep=",", index = False)

In [25]:
dic_new_var = {'name':'13.1 recodage logiciel de traitment',
               'label':'q5_logiciel_traitement_rec',
               'group':'4_logiciel_traitement',
               'personal_data':False,
               'type':'multiple_choice',
               'opened_question':False,
               'type_panda':df0.q5_logiciel_traitement_rec.dtypes,
               'no_question':5,
               'question':df_col.question.loc[df_col.label=="q5_r"].iloc[0],
               "question_family":df_col.question_family.loc[df_col.label=="q5_r"].iloc[0],
               'comment':"Regroupe les questions 5 et 6 sur les logiciels de traitement"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)
df_col

dic_new_var = {'name':'13.2 recodage familles de logiciel',
               'label':'q5_family_logi',
               'group':'4_logiciel_traitement',
               'personal_data':False,
               'type':'multiple_choice',
               'opened_question':False,
               'type_panda':df0.q5_family_logi.dtypes,
               'no_question':5,
               'question':df_col.question.loc[df_col.label=="q5_r"].iloc[0],
               "question_family":df_col.question_family.loc[df_col.label=="q5_r"].iloc[0],
               'comment':"Classe les logiciels par famille"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)
df_col

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
144,110. doctorant,q25_doctorant,12_descript_repondant,False,simple_nominal,False,object,25.0,2/ Êtes-vous ?,NaN,NaN
145,3. octavi_depot_rec,q3_octavi_rec,2_bib_num,False,simple_nominal,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,NaN,NaN
146,14. other_lang_rec,q5_1_other_lang_rec,4_logiciel_traitement,False,multiple_choice,False,object,14.0,"5.1/ Si Autre, précisez lequel (lesquels) ?",q5,Recodage de la question 14. mise en basse cass...
147,13.1 recodage logiciel de traitment,q5_logiciel_traitement_rec,4_logiciel_traitement,False,multiple_choice,False,object,5.0,"5/ Dans le cadre de vos travaux de recherche, ...",q5,Regroupe les questions 5 et 6 sur les logiciel...



## Développement et dépôt de codes ou de logiciels

In [26]:
rec_code={
    "Mes activités de recherche consist à développer des outils spécialisés pour le traitement de la parole et l'analyse des médias":
    {"what":"traitement du son","group":"traitement et analyse de données", "how":""},
    
    "des apps web": 
    {"what":"applications web","group":"programmes et logiciels variés", "how":""},
    
    "Des logiciels variés, allant du materiel d'expérience à des productions directes à être diffusées.":
    {"what":"outils d'expériences", "group":"collecte de données|programmes et logiciels variés", "how":""},
    
    "Questionnaires et programmes d'expérience sur Psytoolkit":
    {"what":"outils d'expérience","group":"collecte de données", "how":"psychtoolkit"},
    
    "Beaucoup d'outils cf. https://samszo.jardindesconnaissances.fr/HDR/cv/cv.html#sec-item299386":
     {"what":"", "group":"programmes et logiciels variés", "how":""},
    
    "Les Commades Stata":
    {"what":"des commandes stata", "group":"traitement et analyse de données", "how":"Stata"},
    
    "Plugin de moteur temps réel pour la XR.":
    {"what":"plugins pour réalité virtuelle", "group":"programmes et logiciels variés", "how":""},
    
    "Je ne l'ai pas fait moi-même mais à grenoble je travaille avec les ingénieurs de la PUD pour fabriqueur un outil semi automatique de pseudonymisation.":
    {"what":"outils de pseudonymisation", "group": "traitement et analyse de données","how":""},
    
    "Une librairie Julia pour du traitement de réseaux.":
    {"what":"analyse de réseau", "group":"traitement et analyse de données", "how":"Julia"},
    
    "plateformes, programmes logiciels, œuvres":
    {"what":"programmes et logiciels variés", "group":"programmes et logiciels variés", "how":""},
    
    "Logiciels de synthèse et traitement du son":
    {"what":"traitement du son","group":"traitement et analyse de données", "how":""},
    
    "Outil de transcription et d'annotation de données": 
    {"what":"transcription et annotation", "group":"traitement et analyse de données", "how":""},
    
    "https://pablo.rauzy.name/software.html":
    {"what":"", "group":"programmes et logiciels variés","how":""},
    
    "création de site internet pour valorisation et diffusion des activités de recherche, productions des étudiants, etc": 
    {"what":"sites web", "group": "diffusion", "how":""},
    
    "Implémentation d's ou algorithme de démonstration":
    {"what":"implémentation", "group": "programmes et logiciels variés", "how":""},
    
    "jeu sérieux sur tablette": 
    {"what":"jeu sérieux", "group":"collecte de données", "how":""},
    
    "Développez les scripts/méthodologies pour traiter mes données":
    {"what":"scripts", "group":"traitement et analyse de données", "how":""},
    
    "Plateforme PatriMaths, sémathèque : https://sematheque.ahp-numerique.fr":
    {"what":"plateforme","group":"diffusion","how":""},
    
    "Dans le cadre du consortium Projets Time Machine, plusieurs logiciels/codes spécialisés sont des logiciels: d'analyse morphologique (MorphAL = Analyse morphologique) qui a énté intégrée comme un plug-in dans QGis ; Amado-on-line (logiciel de visualisation des matrices graphiques le suivant de Bertin)":
    {"what":"analyse morphologique", "group":"traitement et analyse de données", "how":"qgis"},
    
    "prédication de C02 dans Paris":
    {"what":" prédiction", "group":"traitement et analyse de données", "how":""},
    "Scripts Python. En ce moment : prestation avec le CERES pour met à jour Digipower Academy, en partenariat avec l'association suisse Personaldata.io":
    {"what":"applications web", "group":"programmes et logiciels variés","how":"python"},
    
    "jouer aux jeux à 1, à 2 et joueurs et + selon les jeux (ou problèmes considérés)":
    {"what":"jeux", "group":"collecte de données", "how":""},
    
    "Des logiciels d'Interaction Humain-Machine transportante des appareils pétéraux d'interactions (recueil de données de protocoles d'interaction)":
    {"what":"logiciels d'interaction humain-machine", "group":"collecte de données", "how":""},
    
    "Des cahiers Jupyter pour l'analyse des données de recherche":
    {"what":"notebook", "group":"traitement et analyse de données","how":"jupyter"},
    
    "Nous avons avec l'INA une IA (gpt-oss-120) pour thématiser de gros cormus de vidéos":
    {"what":"classification automatique", "group":"traitement et analyse de données", "how":"IA générative"},
    
    "une série d'outil de modélisation à base d'approche profond lerning":
    {"what":"modélisation",  "group":"traitement et analyse de données", "how":"deep learning"},
    
    "Dans un projet de recherche je participe à participer, nous faisons réaliser un HTR de formulaires manuscrits.":
    {"what":"reconnaissance de caractère", "group":"traitement et analyse de données","how":""},
    
    "des code pour les expériences à l'aide d'outils type PsychToolkit":
    {"what":"outils d'expérience", "group":"collecte de données", "how":"psychtoolkit"},
    "TEI/XML adapté à une correspondance du xviiie siècle":
    {"what":"transcription et annotation", "group":"traitement et analyse de données","how":"TEI/XML"},
    
    "Base de de donation TOFLLIT18 (issu d'un projet ANR)":
    {"what":"base de données", "group":"collecte de données", "how":""},
    
    "des scripts R pour du traitement d'analyse de despageses":
    {"what":"scripts", "group":"traitement et analyse de données", "how":""}
}

In [27]:
list_exemple_code = [x for x in df0.q6_1_dev_exemple.loc[~df0.q6_1_dev_exemple.isna()]]
new_rec_code = {}
for n, x in enumerate(rec_code):
    new_key = list_exemple_code[n]
    new_rec_code[new_key]= rec_code[x]

new_rec_code


{"Mes activités de recherche consiste à développer des outils spécialisés pour le traitement de la parole et l'analyse des médias": {'what': 'traitement du son',
  'group': 'traitement et analyse de données',
  'how': ''},
 'des apps web': {'what': 'applications web',
  'group': 'programmes et logiciels variés',
  'how': ''},
 "Des logiciels variés, allant du materiel d'expérience à des productions directe visant à être diffusées.": {'what': "outils d'expériences",
  'group': 'collecte de données|programmes et logiciels variés',
  'how': ''},
 "Questionnaires et programmes d'expérimentaiton sur Psytoolkit": {'what': "outils d'expérience",
  'group': 'collecte de données',
  'how': 'psychtoolkit'},
 "Beaucoup d'outils cf. https://samszo.jardindesconnaissances.fr/HDR/cv/cv.html#sec-item299386": {'what': '',
  'group': 'programmes et logiciels variés',
  'how': ''},
 'Des commades Stata': {'what': 'des commandes stata',
  'group': 'traitement et analyse de données',
  'how': 'Stata'},
 'P

In [28]:
list_row= []
for x in new_rec_code:

    id_row = df0.q45_clé.loc[df0.q6_1_dev_exemple==x].values

    if len(id_row) > 0:
        exemple = new_rec_code[x]
        dict_row = {"q45_clé":id_row[0],
              "q6_1_dev_what":exemple["what"].lower(),
              "q6_1_dev_group":exemple["group"].lower(),
              "q6_1_dev_how": exemple["how"].lower()}
        list_row.append(dict_row)

dftp = pd.DataFrame.from_dict(list_row)
dftp


,q45_clé,q6_1_dev_what,q6_1_dev_group,q6_1_dev_how
0,CN6B-UEHQ,traitement du son,traitement et analyse de données,
1,5G5M-BM3L,applications web,programmes et logiciels variés,
2,BKF8-SNEZ,outils d'expériences,collecte de données|programmes et logiciels va...,
3,YTMC-B4DP,outils d'expérience,collecte de données,psychtoolkit
4,ZGPH-2FPH,,programmes et logiciels variés,
5,SKD4-XJMH,des commandes stata,traitement et analyse de données,stata
6,RVX8-7FSV,plugins pour réalité virtuelle,programmes et logiciels variés,
7,E3YL-PHZV,outils de pseudonymisation,traitement et analyse de données,
8,RAUZ-BRFQ,analyse de réseau,traitement et analyse de données,julia
9,QH7Q-PJZ9,programmes et logiciels variés,programmes et logiciels variés,


In [29]:
df0 = df0.merge(dftp, on=["q45_clé"], how ="left")
df0

,qno_obs,q1_so_principles,q2_hal_depot,q3_octavi_depot,q4_diff_data,q4_autres_entrepots,q5_r,q5_python,q5_excel,q5_julia,...,q54_appareil_utilisé_pour_la_saisie,q25_statut_rec,q25_doctorant,q3_octavi_rec,q5_1_other_lang_rec,q5_logiciel_traitement_rec,q5_family_logi,q6_1_dev_what,q6_1_dev_group,q6_1_dev_how
0,1,"Oui, un peu","Oui, une fois",Non,Non,NaN,Non,Non,Non,Non,...,PC,Doctorant,True,Non,None,None,None,NaN,NaN,NaN
1,2,"Oui, un peu","Oui, une fois",Non,Non,NaN,Non,Non,Non,Non,...,PC,Chercheur,False,Non,None,None,None,NaN,NaN,NaN
2,3,"Oui, tout à fait","Oui, plusieurs fois",Je ne connais pas Octaviana,Non,NaN,Non,Non,Oui,Non,...,PC,Chercheur,False,Non,maxqda|endnotes|zotero,excel|maxqda|endnotes|zotero,feuille de calcul|CAQDAS|gestion biblio,NaN,NaN,NaN
3,4,"Oui, un peu","Oui, une fois",Non,"Autre, précisez",OSF,Oui,Non,Non,Non,...,PC,Chercheur,False,Non,None,r,script et programmation,NaN,NaN,NaN
4,5,"Oui, un peu","Oui, systématiquement",Non,Non,NaN,Non,Non,Oui,Non,...,PC,Chercheur,False,Non,access,excel|access,feuille de calcul|gestion de base de donnée,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,117,"Oui, un peu","Oui, plusieurs fois",Non,"Autre, précisez",OSF,Oui,Oui,Oui,Non,...,PC,Doctorant,True,Non,None,r|python|excel,script et programmation|feuille de calcul,NaN,NaN,NaN
117,118,"Oui, un peu","Oui, systématiquement",Non,Non,NaN,Oui,Non,Oui,Non,...,PC,Chercheur,False,Non,None,r|excel,script et programmation|feuille de calcul,NaN,NaN,NaN
118,119,"Oui, un peu","Oui, plusieurs fois",Non,Non,NaN,Oui,Non,Oui,Non,...,PC,Chercheur,False,Non,None,r|excel,script et programmation|feuille de calcul,scripts,traitement et analyse de données,
119,120,Non,"Oui, plusieurs fois",Non;Je ne connais pas Octaviana,"Autre, précisez",Mendeley data,Non,Oui,Oui,Non,...,PC,Chercheur,False,Non,None,python|excel,script et programmation|feuille de calcul,NaN,NaN,NaN


In [30]:
# on enregistre les nouvelles variables dans le dataframe original
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/{last_file}.csv", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)

In [31]:
dic_new_var = {'name':'16. recodage dev_exemple',
               'label':'q6_1_dev_what',
               'group':'5_developpement',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df0.q6_1_dev_what.dtypes,
               'no_question':16,
               'question':df_col.question.loc[df_col.label=="q6_1_dev_exemple"].iloc[0],
               'comment':"Recodage de la question 16. Restitue le type d'objet développé : traitement du son, jeu, base de donnée, site web"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col,new_variable], ignore_index = True)
df_col1

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
145,3. octavi_depot_rec,q3_octavi_rec,2_bib_num,False,simple_nominal,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,NaN,NaN
146,14. other_lang_rec,q5_1_other_lang_rec,4_logiciel_traitement,False,multiple_choice,False,object,14.0,"5.1/ Si Autre, précisez lequel (lesquels) ?",q5,Recodage de la question 14. mise en basse cass...
147,13.1 recodage logiciel de traitment,q5_logiciel_traitement_rec,4_logiciel_traitement,False,multiple_choice,False,object,5.0,"5/ Dans le cadre de vos travaux de recherche, ...",q5,Regroupe les questions 5 et 6 sur les logiciel...
148,13.2 recodage familles de logiciel,q5_family_logi,4_logiciel_traitement,False,multiple_choice,False,object,5.0,"5/ Dans le cadre de vos travaux de recherche, ...",q5,Classe les logiciels par famille


In [32]:
dic_new_var = {'name':'16. recodage dev_exemple groupe',
               'label':'q6_1_dev_group',
               'group':'5_developpement',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df0.q6_1_dev_group.dtypes,
               'no_question':16,
               'question':df_col.question.loc[df_col.label=="q6_1_dev_exemple"].iloc[0],
               'comment':"Classe les exemples de logiciels développés dans des groupes : outils de traitement et d'analyse, outils de diffusion, collecte de données, etc."
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
146,14. other_lang_rec,q5_1_other_lang_rec,4_logiciel_traitement,False,multiple_choice,False,object,14.0,"5.1/ Si Autre, précisez lequel (lesquels) ?",q5,Recodage de la question 14. mise en basse cass...
147,13.1 recodage logiciel de traitment,q5_logiciel_traitement_rec,4_logiciel_traitement,False,multiple_choice,False,object,5.0,"5/ Dans le cadre de vos travaux de recherche, ...",q5,Regroupe les questions 5 et 6 sur les logiciel...
148,13.2 recodage familles de logiciel,q5_family_logi,4_logiciel_traitement,False,multiple_choice,False,object,5.0,"5/ Dans le cadre de vos travaux de recherche, ...",q5,Classe les logiciels par famille
149,16. recodage dev_exemple,q6_1_dev_what,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Recodage de la question 16. Restitue le type d...


In [33]:
dic_new_var = {'name':'16. recodage dev_exemple how',
               'label':'q6_1_dev_how',
               'group':'5_developpement',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df0.q6_1_dev_how.dtypes,
               'no_question':16,
               'question':df_col.question.loc[df_col.label=="q6_1_dev_exemple"].iloc[0],
               'comment':"Précise le langage ou l'outil de programmation utilisé"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
147,13.1 recodage logiciel de traitment,q5_logiciel_traitement_rec,4_logiciel_traitement,False,multiple_choice,False,object,5.0,"5/ Dans le cadre de vos travaux de recherche, ...",q5,Regroupe les questions 5 et 6 sur les logiciel...
148,13.2 recodage familles de logiciel,q5_family_logi,4_logiciel_traitement,False,multiple_choice,False,object,5.0,"5/ Dans le cadre de vos travaux de recherche, ...",q5,Classe les logiciels par famille
149,16. recodage dev_exemple,q6_1_dev_what,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Recodage de la question 16. Restitue le type d...
150,16. recodage dev_exemple groupe,q6_1_dev_group,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Classe les exemples de logiciels développés da...


In [34]:
df_col1.to_csv("../le_questionnaire/dico_variable.csv", sep=",", index= False)

## Profils répondants

### Recodage autres lieux de travail

In [35]:
for x in df0.q26_1_autre_env.loc[~df0.q26_1_autre_env.isna()]:
    print(x)

Bibliothèque 
Dans les locaux de mes partenaires de recherche
Bibliothèques 
Dans le métro
bibliotheque
Réunions avec les partenaires hospitaliers sur leur site
SITE CNRS
Bibliothèque, archives
terrain 
bureau de collègues dans d'autres universités
Bibliothèque
bibliothèque et archives
bureau hors P8
bibliothèques
Bureau MSH OU BIBLIOTHÈQUE MAIS JE MANQUE CRUELLEMENT D'UN ESPACE DE TRAVAIL 
Où je peux, puisque nous n'avon pas de bureaux. Lorsque je peux, j'évite le campus car : difficultés d'accès, de connexion ; risque que les salles soient déjà prises, etc. Cela est très bloquant pour tout travail d'équipe.
partout
Au laboratoire CEMTI
où je peux, je n'ai pas de bureau à la fac (15 m2 pour 6 titulaires)
colloques, train, hotel
dans des bibliothèque universitaires, au laboratoire  
Transports en commun, salles d'attente...
Lieux de recherche 
Campus Condorcet (chercheur associé à l'Ined)
Bibliothèques P8 et ailleurs
Dans un bureau sur site partagé avec 5 autres collègues, trop petit, 

In [36]:
dict_autre_env ={
    "Bibliothèque" :"Bibliothèque",
"Dans les locaux de mes partenaires de recherche":"Autres sites (Campus Condorcet, CNRS, partenaires)",
"Bibliothèques":"Bibliothèque",
"Dans le métro":"Métro",
"bibliotheque":"Bibliothèque",
"Réunions avec les partenaires hospitaliers sur leur site":"Autres sites (Campus Condorcet, CNRS, partenaires)",
"SITE CNRS":"Autres sites (CNRS, partenaires)",
"Bibliothèque, archives":"Bibliothèque",
"terrain" :"Terrains",
"bureau de collègues dans d'autres universités":"Autres sites (Campus Condorcet, CNRS, partenaires)",
"Bibliothèque":"Bibliothèque",
"bibliothèque et archives":"Bibliothèque",
"bureau hors P8":"Autres sites (CNRS, partenaires)",
"bibliothèques":"Bibliothèque",
"Bureau MSH OU BIBLIOTHÈQUE MAIS JE MANQUE CRUELLEMENT D'UN ESPACE DE TRAVAIL":"Autres sites (Campus Condorcet, CNRS, partenaires)|Bibliothèque",
"Où je peux, puisque nous n'avon pas de bureaux. Lorsque je peux, j'évite le campus car : difficultés d'accès, de connexion ; risque que les salles soient déjà prises, etc. Cela est très bloquant pour tout travail d'équipe.":"Où je peux",
"partout":"Où je peux",
"Au laboratoire CEMTI":"Laboratoire",
"où je peux, je n'ai pas de bureau à la fac (15 m2 pour 6 titulaires)":"Où je peux",
"colloques, train, hotel":"Déplacement",
"dans des bibliothèque universitaires, au laboratoire":"Bibliothèque|Laboratoire",
"Transports en commun, salles d'attente...":"Où je peux",
"Lieux de recherche" :"Où je peux",
"Campus Condorcet (chercheur associé à l'Ined)":"Autres sites (Campus Condorcet, CNRS, partenaires)",
"Bibliothèques P8 et ailleurs":"Bibliothèque|Où je peux",
"Dans un bureau sur site partagé avec 5 autres collègues, trop petit, peu propice à la concentration et pas adapté aux réunions en visio":"Laboratoire",
"Bibliothèques.. Je trouve incongru la question du bureau dont nous ne disposons pas du tout (un peu plus depuis la construction de la Maison de la Recherche mais la chose n'a visiblement pas été pensée jusqu'au bout ou vraiment pour les enseignants chercheurs":"Bibliothèque"
}

In [37]:
df0["q26_1_autre_rec"] = df0.q26_1_autre_env.map(dict_autre_env.get)


In [38]:
# on enregistre les nouvelles variables dans le dataframe original
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/{last_file}.csv", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)

In [39]:
dic_new_var = {'name':'113. recodage Autre, précisez',
               'label':'q26_1_autre_rec',
               'group':'13_env_travail',
               'personal_data':False,
               'type':'nominal_multiple',
               'opened_question':False,
               'type_panda':df0.q26_1_autre_rec.dtypes,
               'no_question':113,
               'question':"Où travaillez-vous le plus souvent? Si autre, précisez",
               'question_family':'q26',
               'comment':"recode les autre lieux"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
148,13.2 recodage familles de logiciel,q5_family_logi,4_logiciel_traitement,False,multiple_choice,False,object,5.0,"5/ Dans le cadre de vos travaux de recherche, ...",q5,Classe les logiciels par famille
149,16. recodage dev_exemple,q6_1_dev_what,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Recodage de la question 16. Restitue le type d...
150,16. recodage dev_exemple groupe,q6_1_dev_group,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Classe les exemples de logiciels développés da...
151,16. recodage dev_exemple how,q6_1_dev_how,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Précise le langage ou l'outil de programmation...


# IA Tools

In [40]:
for x in df0["q35_ia_tools"].loc[~df0.q35_ia_tools.isna()]:
    print(x)

Chat GPT, Mistral IA, LM Notebook
chatgpt, euria
Chat-GPT, Perplexity
le chat ; chatgpt
Intelligence Artificielle désigne de nombreuses technologies. Je n'utilise pas d'outils génératifs mais je travaille avec et développe des outils de transcription, d'analyse de texte et de la parole, d'analyse d'image. Il s'agit de logiciels qui tournent en local sur mon matériel.
hugging face, openAI, gemini, midjourney
Cursor
whisper sur sharedocs
Lumo
Gemini, Ollama, Copilot
DeepL - objection de conscience par rapport à l'IA générative
gemini, chatpgt, claude
chatgpt
Claude, ChatPGT
chatGTP, Perplexity, Emmy
NoScribe en local pour des transcriptions d'entretiens + deepl pour des traductions courtes et ponctuelles.
Deep L, Coco SSD dans le cadre d'un projet de recherche, parfois Chat GPT pour tester les éventuels usages dans des travaux des étudiant.es
chat gpt
ChatGPT et Le CHat
claude ia 
ChatGPT
chatgpt
Chatgpt, deepl
ChatGPT, Gemini, ResearchRabbit, NoteBookLM
Deepl pour la traduction ; Buzz p

In [41]:
#Pour le recodage de la question "q35_ia_tools" on ne prend que les noms d'IA. On traitera les "justifications" avec la qeustion "q37_raison_util_ia"
list_ia = [
    {"reponse":"Chat GPT, Mistral IA, LM Notebook", "ia_name":"chatgpt|mistral|notebooklm"},
    {"reponse":"chatgpt, euria", "ia_name":"chatgpt|euria"},
    {"reponse":"Chat-GPT, Perplexity", "ia_name":"chatgpt|perplexity"},
    {"reponse":"le chat ; chatgpt", "ia_name":"le chat|chatgpt"},
    {"reponse":"Intelligence Artificielle désigne de nombreuses technologies. Je n'utilise pas d'outils génératifs mais je travaille avec et développe des outils de transcription, d'analyse de texte et de la parole, d'analyse d'image. Il s'agit de logiciels qui tournent en local sur mon matériel.","ia_name":""},
    {"reponse":"hugging face, openAI, gemini, midjourney","ia_name":"openai|hugging face|gemini"},
    {"reponse":"Cursor","ia_name":"cursor"},
    {"reponse":"whisper sur sharedocs","ia_name":"whisper"},
    {"reponse":"Lumo","ia_name":"lumo"},
    {"reponse":"Gemini, Ollama, Copilot", "ia_name":"gemini|ollama|copilot"},
    {"reponse":"DeepL - objection de conscience par rapport à l'IA générative","ia_name":"deepl"},
    {"reponse":"gemini, chatpgt, claude", "ia_name":"gemini|chatgpt|claude"},
    {"reponse":"chatgpt", "ia_name":"chatgpt"},
    {"reponse":"Claude, ChatPGT", "ia_name":"claude|chatgpt"},
    {"reponse":"chatGTP, Perplexity, Emmy", "ia_name":"chatgpt|emmy|perplexity"},
    {"reponse":"NoScribe en local pour des transcriptions d'entretiens + deepl pour des traductions courtes et ponctuelles.","ia_name":"whisper|deepl"},
    {"reponse":"Deep L, Coco SSD dans le cadre d'un projet de recherche, parfois Chat GPT pour tester les éventuels usages dans des travaux des étudiant.es","ia_name":"deepl|coco ssd|chatgpt"},
    {"reponse":"chat gpt","ia_name":"chatgpt"},
    {"reponse":"ChatGPT et Le CHat","ia_name":"chatgpt|le chat"},
    {"reponse":"claude ia","ia_name":"claude"},
    {"reponse":"ChatGPT","ia_name":"chatgpt"},
    {"reponse":"chatgpt","ia_name":"chatgpt"},
    {"reponse":"Chatgpt, deepl", "ia_name":"chatgpt"},
    {"reponse":"ChatGPT, Gemini, ResearchRabbit, NoteBookLM","ia_name":"chatgpt|gemini|researchrabbit|notebooklm"},
    {"reponse":"Deepl pour la traduction ; Buzz pour la retranscription", "ia_name":"deepl|buzz"},
    {"reponse":"Claude", "ia_name":"claude"},
    {"reponse":"Antidote","ia_name":"antidote"},
    {"reponse":"chatgpt, claude, perplexity...", "ia_name":"chatgpt|claude|perplexity"},
    {"reponse":"ChatGPT, Firefly", "ia_name":"chatgpt|firefly"},
    {"reponse":"Anara, Notebook lm, chat gpt, antidote","ia_name":"anara|notebooklm|chatgpt|antidote"},
    {"reponse":"Claude", "ia_name":"claude"},
    {"reponse":"très peu ; chat GPT mais je l'utilise très peu car ce n'est pas un compte pro; je préfère attendre que l'on ait des recommandations en terme d'utilisation de l'IA par l'université, avec peut être un compte pro pour protéger le contenu dépose ; j'utilise par contre deep pour une aide à la traduction","ia_name":"chatgpt|deepl"},
    {"reponse":"ChatGPT","ia_name":"chatgpt"},
    {"reponse":"ChatGPT, Claude, outils spécialisés (génération sonore : architectures Rave)","ia_name":"chatgpt|claude|outils spécialisés"},
    {"reponse":"Chatgpt","ia_name":"chatgpt"},
    {"reponse":"Chat GPT","ia_name":"chatgpt"},
    {"reponse":"ChatGPT, Gemini, claude","ia_name":"chatgpt|gemini|claude"},
    {"reponse":"Claude / ChatGPT","ia_name":"claude|chatgpt"},
    {"reponse":"Perplexity","ia_name":"perplexity"},
    {"reponse":"Chat GPT","ia_name":"chatgpt"},
    {"reponse":"mistral, whisper","ia_name":"mistral|whisper"},
    {"reponse":"ChatGPT, Gemini, ....","ia_name":"chatgpt|gemini"},
    {"reponse":"ChatGPT","ia_name":"chatgpt"},
    {"reponse":"ChatGPT, Claude, Sigmund","ia_name":"chatgpt|claude|sigmund"},
    {"reponse":"Mistral", "ia_name":"mistral|chatgpt"},
    {"reponse":"traduction (Deepl)","ia_name":"deepl"},
    {"reponse":"IA gratuites", "ia_name":""},
    {"reponse":"Claude, Elicit", "ia_name":"claude|elicit"},
    {"reponse":"Mistral, Chat GPT, Claude","ia_name":"mistral|chatgpt|claude"},
    {"reponse":"chatgpt, Claude, etc.", "ia_name":"chatgpt|claude"},
    {"reponse":"ChatGPT","ia_name":"chatgpt"},
    {"reponse":"ChatGPT, OpenAI (recherche)", "ia_name":"openai|chatgpt"},
    {"reponse":"Chat GPT, Perplexity","ia_name":"chatgpt|perplexity"},
    {"reponse":"GPT, gpt-oss-120, DeepSeek", "ia_name":"chatgpt|gpt-oss-120|deepseek"},
    {"reponse":"Claude et ChatGPT", "ia_name":"claude|chatgpt"},
    {"reponse":"Claude Desktop, LeChat, SciSpace", "ia_name":"claude|le chat|scispace"},
    {"reponse":"Je participe à deux projets de recherche qui utilisent ce type d'outil","ia_name":""},
    {"reponse":"Mistral, Gemini, Copilot, Chatgpt, DeepL", "ia_name":"mistral|gemini|copilot|chatgpt|deepl"},
    {"reponse":"Claude", "ia_name":"claude"},
    {"reponse":"Emmy","ia_name":"emmy"},
    {"reponse":"Perplexity, Claude, ChatGPT","ia_name":"perplexity|claude|chatgpt"},
    {"reponse":"Copilot, Deepseek, Whisper", "ia_name":"copilot|deepseek|whisper"},
    {"reponse":"Chat GPT,  IA associée aux App (dont Apple IA)","ia_name":"chatgpt|apple ia"},
    {"reponse":"Chat GPT ; Claude","ia_name":"chatgpt|claude"},
    {"reponse":"Elicit","ia_name":"elicit"},
    {"reponse":"mistral","ia_name":"mistral"},
    {"reponse":"Claude, ChatGPT", "ia_name":"claude|chatgpt"},
    {"reponse":"Correcteurs d'orthographe","ia_name":"correcteurs d'orthographe"}
]


In [42]:
for n, x in enumerate(df0["q45_clé"].loc[~df0.q35_ia_tools.isna()]):
    dict_ia= list_ia[n]
    dict_ia["q45_clé"]=x

In [43]:
df_ia = pd.DataFrame.from_dict(list_ia)
df01 = df0.merge(df_ia[["q45_clé","ia_name"]].rename(columns={"ia_name":"q35_ia_tools_rec"}), on =["q45_clé"], how ="left")


In [44]:

# on enregistre les nouvelles variables dans le dataframe original
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/{last_file}", "w") as file_out:
    df01.to_csv(file_out, sep=",", index= False)

In [45]:
dic_new_var = {'name':'135. recodage ia_tools',
               'label':'q35_ia_tools',
               'group':'4_logiciel_traitement',
               'personal_data':False,
               'type':'nominal_multiple',
               'opened_question':False,
               'type_panda':df01.q35_ia_tools_rec.dtypes,
               'no_question':135,
               'question':"Si oui, lesquels ?",
               'question_family':'q35',
               'comment':"recode les outils ia"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
149,16. recodage dev_exemple,q6_1_dev_what,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Recodage de la question 16. Restitue le type d...
150,16. recodage dev_exemple groupe,q6_1_dev_group,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Classe les exemples de logiciels développés da...
151,16. recodage dev_exemple how,q6_1_dev_how,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Précise le langage ou l'outil de programmation...
152,"113. recodage Autre, précisez",q26_1_autre_rec,13_env_travail,False,nominal_multiple,False,object,113.0,"Où travaillez-vous le plus souvent? Si autre, ...",q26,recode les autre lieux


In [26]:
df_col1.to_csv("../le_questionnaire/dico_variable.csv", index=False)